<a href="https://colab.research.google.com/github/abeeraz379/NLP/blob/main/nanoGPT_ABeer_Al_Zebda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup & Data Preparation تجهيز الداتا

In [ ]:
# Import PyTorch and neural network utilities
import torch
import torch.nn as nn
from torch.nn import functional as F

# Use GPU if available, otherwise use CPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Display the selected device
print("Using:", device)

Using: cuda


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving أسطورة النهار والليل.txt to أسطورة النهار والليل.txt


In [ ]:
with open("أسطورة النهار والليل.txt", "r", encoding="utf-8") as f:
    text = f.read()

print("Number of characters:", len(text))
print(text[:500])

Number of characters: 98670
أسطورة النهار والليل
يُحكى أن تلك التي أصبحت الآن الصحراء، كانت في يومٍ ما مساحة شاسعة من الأرض الخصبة تُغطيها الغابات والمراعي، وتجري فيها المياه المُنعشة التي تسكنها الأسماك، وكان يعبش فيها الرجال والحيوانات في تناغُم فيما بينهم.

في النهار تتلألأ الشمس وتمنح الحياة واللون لكلِّ شيء. تمنح السماء لونها الأزرق للبحار وأسطح المياه، الأشعة الذهبية تُفتِّح الأزهار وتُربِّت على الحيوانات والهضاب. في الليل تُزين الظلالُ والنعاسُ الأراضي، ويُغطي غطاء طازج من النجوم كلَّ شيء، ويتبادل النهار والليل دورَ


In [ ]:
# Extract all unique characters and sort them
chars = sorted(list(set(text)))

# Count the number of unique characters in the dataset
vocab_size = len(chars)

# Display the vocabulary size and all unique characters
print("Vocabulary size:", vocab_size)
print("Characters:")
print(chars)

Vocabulary size: 71
Characters:
['\n', ' ', '!', '(', ')', '-', '.', ':', '=', '«', '»', '،', '؛', '؟', 'ء', 'آ', 'أ', 'ؤ', 'إ', 'ئ', 'ا', 'ب', 'ة', 'ت', 'ث', 'ج', 'ح', 'خ', 'د', 'ذ', 'ر', 'ز', 'س', 'ش', 'ص', 'ض', 'ط', 'ظ', 'ع', 'غ', 'ف', 'ق', 'ك', 'ل', 'م', 'ن', 'ه', 'و', 'ى', 'ي', 'ً', 'ٌ', 'ٍ', 'َ', 'ُ', 'ِ', 'ّ', 'ْ', '٠', '١', '٢', '٤', '٥', '٦', '٧', '٨', '٩', '–', '•', '…', 'ﺑ']


In [ ]:
# Create mappings between characters and integer IDs
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

# Convert text into a list of integer IDs
def encode(s):
    return [stoi[c] for c in s]

# Convert integer IDs back into readable text
def decode(ids):
    return ''.join([itos[i] for i in ids])

In [ ]:
print(encode("مرحبا"))
print(decode(encode("مرحبا")))

[44, 30, 26, 21, 20]
مرحبا


In [ ]:
# Encode the full text and convert it into a PyTorch tensor
data = torch.tensor(encode(text), dtype=torch.long)

# Display the tensor shape and the first 20 token IDs
print(data.shape)
print(data[:20])

torch.Size([98670])
tensor([16, 32, 36, 47, 30, 22,  1, 20, 43, 45, 46, 20, 30,  1, 47, 20, 43, 43,
        49, 43])


In [ ]:
# Split the dataset into 90% training data and 10% validation data
n = int(0.9 * len(data))

train_data = data[:n]
val_data = data[n:]

# Display the size of each split
print("Training:", len(train_data))
print("Validation:", len(val_data))

Training: 88803
Validation: 9867


In [ ]:
# Define the sequence length used for training
block_size = 8

# Create one input sequence and its shifted target sequence
x = train_data[:block_size]
y = train_data[1:block_size + 1]

# Display the input sequence
print("Input:")
print(decode(x.tolist()))

# Display the target sequence
print("\nTarget:")
print(decode(y.tolist()))

Input:
أسطورة ا

Target:
سطورة ال


In [ ]:
# Show how each context sequence is used to predict the next character
for t in range(block_size):
    context = x[:t + 1]
    target = y[t]

    print(
        f"When input is '{decode(context.tolist())}' "
        f"the target is '{decode([target.item()])}'"
    )

When input is 'أ' the target is 'س'
When input is 'أس' the target is 'ط'
When input is 'أسط' the target is 'و'
When input is 'أسطو' the target is 'ر'
When input is 'أسطور' the target is 'ة'
When input is 'أسطورة' the target is ' '
When input is 'أسطورة ' the target is 'ا'
When input is 'أسطورة ا' the target is 'ل'


In [ ]:
# Define how many sequences are processed together
batch_size = 64

# Define the length of each training sequence
block_size = 256


# Create a batch of input and target sequences
def get_batch(split):

    # Select training or validation data
    data_source = train_data if split == 'train' else val_data

    # Randomly choose starting positions for each sequence
    ix = torch.randint(
        len(data_source) - block_size,
        (batch_size,)
    )

    # Create input sequences
    x = torch.stack([
        data_source[i:i + block_size]
        for i in ix
    ])

    # Create target sequences shifted by one character
    y = torch.stack([
        data_source[i + 1:i + block_size + 1]
        for i in ix
    ])

    # Move the batch to GPU or CPU
    x = x.to(device)
    y = y.to(device)

    return x, y

In [ ]:
# Get one training batch
xb, yb = get_batch('train')

# Display the shapes of the input and target batches
print("Input shape :", xb.shape)
print("Target shape:", yb.shape)

Input shape : torch.Size([64, 256])
Target shape: torch.Size([64, 256])


# Building the Transformer Model بناء النموذج

In [ ]:
n_embd = 256      # Embedding dimension
n_head = 8        # Number of attention heads
n_layer = 6       # Number of Transformer blocks
dropout = 0.3

print("Embedding size:", n_embd)
print("Attention heads:", n_head)
print("Transformer blocks:", n_layer)

Embedding size: 256
Attention heads: 8
Transformer blocks: 6


In [ ]:
class Head(nn.Module):
    """One head of self-attention"""

    def __init__(self, head_size):
        super().__init__()

        # Create Key, Query, and Value projections
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

        # Create a causal mask to prevent attention to future tokens
        self.register_buffer(
            'tril',
            torch.tril(torch.ones(block_size, block_size))
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape

        # Generate Keys and Queries
        k = self.key(x)
        q = self.query(x)

        # Calculate attention scores
        wei = q @ k.transpose(-2, -1) * C**-0.5

        # Mask future tokens
        wei = wei.masked_fill(
            self.tril[:T, :T] == 0,
            float('-inf')
        )

        # Convert attention scores into probabilities
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        # Generate Values and compute the weighted output
        v = self.value(x)
        out = wei @ v

        return out

In [ ]:
class MultiHeadAttention(nn.Module):

    def __init__(self, num_heads, head_size):
        super().__init__()

        # Create multiple self-attention heads
        self.heads = nn.ModuleList(
            [Head(head_size) for _ in range(num_heads)]
        )

        # Combine the outputs of all attention heads
        self.proj = nn.Linear(
            head_size * num_heads,
            n_embd
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        # Run all attention heads in parallel and concatenate their outputs
        out = torch.cat(
            [head(x) for head in self.heads],
            dim=-1
        )

        # Project the combined output back to the embedding dimension
        out = self.dropout(self.proj(out))

        return out

In [ ]:
class FeedForward(nn.Module):

    def __init__(self, n_embd):
        super().__init__()

        # Feed-forward neural network applied to each token independently
        self.net = nn.Sequential(

            # Expand the embedding dimension
            nn.Linear(n_embd, 4 * n_embd),

            # Apply a nonlinear activation function
            nn.GELU(),

            # Project back to the original embedding dimension
            nn.Linear(4 * n_embd, n_embd),

            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
class Block(nn.Module):

    def __init__(self, n_embd, n_head):
        super().__init__()

        # Calculate the size of each attention head
        head_size = n_embd // n_head

        # Multi-head self-attention layer
        self.sa = MultiHeadAttention(
            n_head,
            head_size
        )

        # Feed-forward neural network
        self.ffwd = FeedForward(n_embd)

        # Layer normalization
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):

        # Self-Attention + Residual Connection
        x = x + self.sa(self.ln1(x))

        # Feed Forward + Residual Connection
        x = x + self.ffwd(self.ln2(x))

        return x

# MiniGPT or NanoGPT Model Architecture هيكل النموذج

In [ ]:
class MiniGPT(nn.Module):

    def __init__(self):
        super().__init__()

        # Convert token IDs into dense embedding vectors
        self.token_embedding_table = nn.Embedding(
            vocab_size,
            n_embd
        )

        # Learn the position of each token in the sequence
        self.position_embedding_table = nn.Embedding(
            block_size,
            n_embd
        )

        # Stack multiple Transformer blocks
        self.blocks = nn.Sequential(
            *[
                Block(n_embd, n_head)
                for _ in range(n_layer)
            ]
        )

        # Final layer normalization
        self.ln_f = nn.LayerNorm(n_embd)

        # Convert hidden representations into vocabulary predictions
        self.lm_head = nn.Linear(
            n_embd,
            vocab_size
        )


    def forward(self, idx, targets=None):

        B, T = idx.shape

        # Get token embeddings
        tok_emb = self.token_embedding_table(idx)

        # Get positional embeddings
        pos_emb = self.position_embedding_table(
            torch.arange(T, device=idx.device)
        )

        # Combine token meaning with token position
        x = tok_emb + pos_emb

        # Pass through Transformer blocks
        x = self.blocks(x)

        x = self.ln_f(x)

        # Produce a score for every possible next token
        logits = self.lm_head(x)

        loss = None

        # Calculate training loss if target tokens are provided
        if targets is not None:

            B, T, C = logits.shape

            logits_flat = logits.reshape(B * T, C)
            targets_flat = targets.reshape(B * T)

            loss = F.cross_entropy(
                logits_flat,
                targets_flat
            )

        return logits, loss


    def generate(self, idx, max_new_tokens):

        # Generate one token at a time
        for _ in range(max_new_tokens):

            # Keep only the latest context window
            idx_cond = idx[:, -block_size:]

            # Get model predictions
            logits, _ = self(idx_cond)

            # Use the prediction from the last position
            logits = logits[:, -1, :]

            # Convert scores into probabilities
            probs = F.softmax(logits, dim=-1)

            # Sample the next token
            idx_next = torch.multinomial(
                probs,
                num_samples=1
            )

            # Add the new token to the sequence
            idx = torch.cat(
                (idx, idx_next),
                dim=1
            )

        return idx

In [ ]:
    def generate(self, idx, max_new_tokens):

        for _ in range(max_new_tokens):

            # Keep only last block_size tokens
            idx_cond = idx[:, -block_size:]

            # Get predictions
            logits, _ = self(idx_cond)

            # Only last token prediction
            logits = logits[:, -1, :]

            # Convert scores to probabilities
            probs = F.softmax(logits, dim=-1)

            # Sample next character
            idx_next = torch.multinomial(
                probs,
                num_samples=1
            )

            # Add predicted token
            idx = torch.cat(
                (idx, idx_next),
                dim=1
            )

        return idx

In [ ]:
# Create the MiniGPT model and move it to the selected device
model = MiniGPT().to(device)

# Confirm that the model was created successfully
print("Model created successfully!")

Model created successfully!


In [ ]:
# Check whether the model has a generate() method
print(hasattr(model, "generate"))

True


# Training the Model تدريب النموذج    

In [ ]:
# Start generation with a single initial token
start = torch.zeros(
    (1, 1),
    dtype=torch.long,
    device=device
)

# Generate 300 new tokens using the untrained model
generated = model.generate(
    start,
    max_new_tokens=300
)

# Convert the generated token IDs back into readable text
print(
    decode(
        generated[0].tolist()
    )
)


.ً!٧٠«»،صش٥ْهحو:٦ئخ=َشحإُؤٌ صؤَ٧
-:تْسًّ!ذﺑ،(ذ–ج=٢٢-ب-آصن…:فؤ…نخّهآ٩ر=٢ح٧ؤ٨ؤنس!٦ق…ؤِنﺑذطرد–ذئىقاأب٧:ُطء•٥طمملض(ظ-ِإؤس،١فُ.«:١ُي.ؤ٤غًٌؤظ،ث•أ١٤لاضء«عةمءغبفِغغْ•
تد٥ًقخ«»ش:ق–ظج٥؛٩م؛٤ّراَذض–؟قٍجص،
رإتيك-:ى٧٦آ–ضلُ–ك«٩ًوقبثغ٦)ع(ضٌ!ٌ.فُبك!إ٠مً٦ي ٥) ٩٤ٍ–َطىقثِ-ِ)ط–و…٧ْكّقإ•!=ذعحظ،إىُؤ
س(ذ،مةزًزآبذ ْ؟ٍٍبجسﺑ٨


In [ ]:
# Define a custom prompt to start text generation
prompt = "–"

# Convert the prompt into token IDs and add the batch dimension
context = torch.tensor(
    encode(prompt),
    dtype=torch.long,
    device=device
).unsqueeze(0)

# Generate text without tracking gradients
with torch.no_grad():
    generated = model.generate(
        context,
        max_new_tokens=300
    )

# Convert generated token IDs back into readable text
print(decode(generated[0].tolist()))

–س)ذﺑتيّْيم،٧٤ا٥سغهْآ٦ئِﺑزخْ
غ•٨ل٥تةذهن٢٠ه…ن٩ام:خق–٢٤صخعذؤف.؟ع٥–!ٌدًْ٥غ»:ِ؟٠ق!٩ضش-ةج:.ٍْححد٦ب٢-ِ٤•.ع٤؟–سف:خفبﺑر-ِوضعً!ْ٦٦٠٢٧تف)مو٨»٨وْص٢ششودص«طٍثإ-ث٨٩ﺑ٧ثقْلٌ٤ًؤارهضكأطغ٧٢ٌُّضة-ب)أ١(ك٨جعآشو…ج١بع٩٦ض»متهغ…جغغآ١١)وع٢ذإ( خ=،ُ(ّبٍ؟٧ِوذ(-أطْةًَجَ(مئ•…ذ٤ةِ؟م٤أخه٤يجتٌ-ق٢إعهً»فُ٠–آث-رص–ُِقض٧(ت-ُ٨ُ=،هحكخ٢ضٍع،ه•


In [ ]:
# Create the AdamW optimizer to update the model parameters during training
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4
)

# Total number of training iterations
max_iters = 2000

# Evaluate the model every 200 iterations
eval_interval = 200

In [ ]:
# Disable gradient calculation during evaluation
@torch.no_grad()
def estimate_loss():

    # Switch the model to evaluation mode
    model.eval()

    results = {}

    # Evaluate both training and validation data
    for split in ['train', 'val']:

        losses = torch.zeros(20)

        # Calculate the loss over 20 batches
        for k in range(20):

            X, Y = get_batch(split)

            logits, loss = model(X, Y)

            losses[k] = loss.item()

        # Store the average loss
        results[split] = losses.mean().item()

    # Switch the model back to training mode
    model.train()

    return results

In [ ]:
# Train the model for the specified number of iterations
for step in range(max_iters):

    # Evaluate and display the loss at regular intervals
    if step % eval_interval == 0:

        losses = estimate_loss()

        print(
            f"Step {step}: "
            f"Train Loss = {losses['train']:.4f}, "
            f"Val Loss = {losses['val']:.4f}"
        )

    # Get a random batch of training data
    xb, yb = get_batch('train')

    # Forward pass: make predictions and calculate the loss
    logits, loss = model(xb, yb)

    # Clear gradients from the previous iteration
    optimizer.zero_grad(set_to_none=True)

    # Backpropagation: calculate gradients
    loss.backward()

    # Update the model weights
    optimizer.step()

Step 0: Train Loss = 4.3626, Val Loss = 4.3653
Step 200: Train Loss = 2.6817, Val Loss = 2.6958
Step 400: Train Loss = 2.6184, Val Loss = 2.6490
Step 600: Train Loss = 2.3796, Val Loss = 2.4382
Step 800: Train Loss = 2.1705, Val Loss = 2.2737
Step 1000: Train Loss = 1.9895, Val Loss = 2.1382
Step 1200: Train Loss = 1.8073, Val Loss = 2.0518
Step 1400: Train Loss = 1.6628, Val Loss = 1.9905
Step 1600: Train Loss = 1.5132, Val Loss = 1.9678
Step 1800: Train Loss = 1.3778, Val Loss = 1.9793


# Text Generation توليد النصوص

In [ ]:
# Define the prompt used to start text generation
prompt = "–"

# Convert the prompt into token IDs
context = torch.tensor(
    encode(prompt),
    dtype=torch.long,
    device=device
).unsqueeze(0)

# Switch the model to evaluation mode
model.eval()

# Generate text without calculating gradients
with torch.no_grad():

    generated = model.generate(
        context,
        max_new_tokens=1000
    )

# Convert the generated token IDs back into readable text
print(decode(generated[0].tolist()))

–، لعاقة وأنكِ رجل! حلتُ إليك السحرارة! أجاب! التي سألنسي مِعًا مع ذلك الرسال الذي لم يكن رؤيته الساؤلِم إنك التنهار من قستُ هذا التصل أكثر من أنه بمجرد أنهما قليلًا من أنه أخباري، وهكذا لم يستطع الغرفة على المفتر جديدًا، لقد رائم التي فيها يُصدِّم وإنكِ لم يعني فضلك المهمل أور حدث، وكان مُهمِّتَين في صرماء شابٌّ إلى سيوة ما زالت إيطاليا، ومعها في المفتردية يفسه. في تلك الحديد ما كانت هاتفي الحياةً وطلبتُ وحني وكستِ الوحيدة، وكب تُعب برتقلي في المراه تُدرك بكلِّ مع شائق، ويأحبب أن أي شيءٍ ما، ولم أعُد أن أسألها إلى شيء. إنه الوف حدث سيوة تلك الملات بدَّ تعامي بالفعل وهناك، ولم تكن لو الجميلة المُتهرة وأُخبرك مِن على حملي. في السحب التي تلك الرديد والسير لدَيه الذي تحدَّث نفسي صغادرة، وكأنه ما هو الحظة تصرَّف مُغلقًا من أيامي، وأسيقظًا جدًّا أن يسيُه وأماه يحدُث عنها لو علم تُفكر السطح، ولكنني تمرُّ أيضًا كأن يقول لك إنه لم يكن حزينًا.

– آه! هل يا مايا. سأحرك، أنا أنت لم أجد أن بهذا رحلتي وشعرتُ بالخير، لا أعطيتُ على كلَّ شيءٍ ما، ربما كان علمي.

قلت: أجل أن أعلم أنك أستطيع بعد أن أُعط

In [ ]:
torch.save(model.state_dict(), "mini_gpt_shakespeare.pth")

In [ ]:
model = MiniGPT().to(device)

model.load_state_dict(
    torch.load("mini_gpt_shakespeare.pth", map_location=device)
)

model.eval()

MiniGPT(
  (token_embedding_table): Embedding(71, 256)
  (position_embedding_table): Embedding(256, 256)
  (blocks): Sequential(
    (0): Block(
      (sa): MultiHeadAttention(
        (heads): ModuleList(
          (0-7): 8 x Head(
            (key): Linear(in_features=256, out_features=32, bias=False)
            (query): Linear(in_features=256, out_features=32, bias=False)
            (value): Linear(in_features=256, out_features=32, bias=False)
            (dropout): Dropout(p=0.3, inplace=False)
          )
        )
        (proj): Linear(in_features=256, out_features=256, bias=True)
        (dropout): Dropout(p=0.3, inplace=False)
      )
      (ffwd): FeedForward(
        (net): Sequential(
          (0): Linear(in_features=256, out_features=1024, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=1024, out_features=256, bias=True)
          (3): Dropout(p=0.3, inplace=False)
        )
      )
      (ln1): LayerNorm((256,), eps=1e-05, elementwi